# CS 6120 — Exam Practice

Linear algebra, gradients, and word embeddings. Each question has a hidden solution — click **Show solution** to reveal. NumPy code cells let you explore the concepts interactively.

## Contents
- **Part A** — Singular Value Decomposition
- **Part B** — Word2Vec Negative Sampling
- **Part C** — Foundations (dot products, sigmoid, chain rule)
- **Part D** — Eigenvectors and their connection to word embeddings
- **Part E** — NumPy shape and transformation practice

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Part A — NumPy Shape and Transformation Practice

For each question, **predict the shape before running the code**, then verify in the code cell.

### Q31 — Basic shapes
```python
a = np.array([1, 2, 3])
b = np.array([[1], [2], [3]])
c = np.array([[1, 2, 3]])
```
- (a) Shapes of `a`, `b`, `c`?
- (b) Which are 1D vs 2D?
- (c) What does `a.T` produce?

In [ ]:
a = np.array([1, 2, 3])
b = np.array([[1], [2], [3]])
c = np.array([[1, 2, 3]])
print('a:', a.shape, '  a.T:', a.T.shape, '  (unchanged — 1D has no transpose)')
print('b:', b.shape, '  b.T:', b.T.shape)
print('c:', c.shape, '  c.T:', c.T.shape)

<details>
<summary><b>Show solution</b></summary>

- (a) `a.shape == (3,)`, `b.shape == (3, 1)`, `c.shape == (1, 3)`.
- (b) `a` is 1D (one axis). `b` and `c` are 2D (two axes), even though one dimension is 1.
- (c) `a.T` is still `(3,)` — **transposing a 1D array is a no-op**. To get a column vector from `a`, use `a[:, None]` or `a.reshape(-1, 1)`.

</details>

### Q32 — Reshaping
`x = np.arange(12)`.
- (a) Reshape to 3×4.
- (b) Reshape to 4×3. Is this the transpose of (a)?
- (c) What does `x.reshape(-1, 2)` give? What does `-1` mean?

In [ ]:
x = np.arange(12)
print('3x4:\n', x.reshape(3, 4))
print('\n4x3:\n', x.reshape(4, 3))
print('\nIs 4x3 the transpose of 3x4? ', np.array_equal(x.reshape(3, 4).T, x.reshape(4, 3)))
print('\nx.reshape(-1, 2) shape:', x.reshape(-1, 2).shape)

<details>
<summary><b>Show solution</b></summary>

- (a) `x.reshape(3, 4)`
- (b) `x.reshape(4, 3)` — **not** the transpose. Both lay out elements row-by-row from the same flat list; the transpose would reorder elements (`[0, 4, 8, 1, 5, 9, ...]`).
- (c) `x.reshape(-1, 2)` → shape `(6, 2)`. The `-1` means "infer this dimension so the total matches" — NumPy computes `12 / 2 = 6`.

</details>

### Q33 — Broadcasting
```python
A = np.ones((4, 3))
b = np.array([1, 2, 3])        # (3,)
c = np.array([[1], [2], [3], [4]])  # (4, 1)
```
- (a) Shape of `A + b`?
- (b) Shape of `A + c`?
- (c) What happens with `A + np.array([1, 2])` (shape `(2,)`)?

In [ ]:
A = np.ones((4, 3))
b = np.array([1, 2, 3])
c = np.array([[1], [2], [3], [4]])
print('A + b:', (A + b).shape)
print('A + c:', (A + c).shape)
try:
    (A + np.array([1, 2])).shape
except ValueError as e:
    print('A + (2,) error:', e)

<details>
<summary><b>Show solution</b></summary>

- (a) `(4, 3)` — `b` of shape `(3,)` aligns with the **last axis** of `A` and is broadcast across the 4 rows.
- (b) `(4, 3)` — `c` of shape `(4, 1)` broadcasts across the 3 columns.
- (c) **Error**: `(2,)` tries to align with the last axis of `A` (size 3). 2 ≠ 3 and neither is 1, so broadcasting fails.

**Rule**: shapes align from the right; each pair of axes must be equal or one of them must be 1.

</details>

### Q34 — Transpose and matmul
`W` shape `(5, 3)`, `x` shape `(3,)`.
- (a) `W @ x`?
- (b) `W.T @ W`?
- (c) `np.outer(x, x)`?
- (d) To compute $W x x^\top$, what shapes line up?

In [ ]:
W = np.random.randn(5, 3)
x = np.random.randn(3)
print('W @ x:', (W @ x).shape)
print('W.T @ W:', (W.T @ W).shape)
print('np.outer(x, x):', np.outer(x, x).shape)
print('W @ np.outer(x, x):', (W @ np.outer(x, x)).shape)

<details>
<summary><b>Show solution</b></summary>

- (a) `(5,)` — `(5,3) @ (3,) = (5,)`.
- (b) `(3, 3)` — `(3,5) @ (5,3) = (3,3)`.
- (c) `(3, 3)` — outer product of a length-3 vector with itself.
- (d) First compute $x x^\top$ (outer, shape `(3, 3)`), then $W \cdot (xx^\top)$: `(5,3) @ (3,3) = (5,3)`.

</details>

### Q35 — Axis arguments
`A = np.random.randn(10, 5)`.
- (a) Shape of `A.sum(axis=0)`?
- (b) Shape of `A.sum(axis=1)`?
- (c) `A.mean(axis=0, keepdims=True)` and why `keepdims`?

In [ ]:
A = np.random.randn(10, 5)
print('A.sum(axis=0):', A.sum(axis=0).shape, '  (sums down each column)')
print('A.sum(axis=1):', A.sum(axis=1).shape, '  (sums across each row)')
print('A.mean(axis=0):', A.mean(axis=0).shape)
print('A.mean(axis=0, keepdims=True):', A.mean(axis=0, keepdims=True).shape)
# keepdims lets you broadcast directly:
centered = A - A.mean(axis=0, keepdims=True)
print('centered shape:', centered.shape)

<details>
<summary><b>Show solution</b></summary>

- (a) `(5,)` — `axis=0` sums down each column (collapses the 10).
- (b) `(10,)` — `axis=1` sums across each row (collapses the 5).
- (c) `(1, 5)` instead of `(5,)`. **`keepdims=True`** preserves the reduced axis as length 1, which makes the result broadcast cleanly back against the original array (e.g., for centering: `A - A.mean(axis=0, keepdims=True)`).

</details>

### Q36 — Column vs row vector matmul
`v = np.array([1, 2, 3])`, shape `(3,)`.
- (a) Shape of `v @ v`?
- (b) `v[:, None] @ v[None, :]`? What operation?
- (c) `v[None, :] @ v[:, None]`? What operation?

In [ ]:
v = np.array([1, 2, 3])
print('v @ v:', (v @ v), '  (scalar — inner product)')
print('v[:, None] @ v[None, :]:')
print(v[:, None] @ v[None, :])  # outer product
print('shape:', (v[:, None] @ v[None, :]).shape)
print('\nv[None, :] @ v[:, None]:')
print(v[None, :] @ v[:, None])  # inner product as 1x1 matrix
print('shape:', (v[None, :] @ v[:, None]).shape)

<details>
<summary><b>Show solution</b></summary>

- (a) Scalar — the inner product $v \cdot v = 1 + 4 + 9 = 14$.
- (b) `(3, 3)` — outer product $v v^\top$.
- (c) `(1, 1)` — the inner product packaged as a $1 \times 1$ matrix.

</details>

### Q37 — Indexing a batch
`X` shape `(1000, 50)`.
- (a) Grab examples 0, 2, 5, 7. Shape?
- (b) Keep only the first 10 dimensions. Shape?
- (c) Per-example L2 norm. Shape? How?

In [ ]:
X = np.random.randn(1000, 50)
sub = X[[0, 2, 5, 7]]
print('fancy index:', sub.shape)
print('first 10 dims:', X[:, :10].shape)
norms = np.linalg.norm(X, axis=1)
print('per-row norm:', norms.shape)

<details>
<summary><b>Show solution</b></summary>

- (a) `X[[0, 2, 5, 7]]` → `(4, 50)` — fancy indexing with a list of row indices.
- (b) `X[:, :10]` → `(1000, 10)`.
- (c) `np.linalg.norm(X, axis=1)` → `(1000,)` — one L2 norm per row.

</details>

### Q38 — Common bugs
For each, say whether it works and what the result/error is.

(a) `np.array([1, 2, 3]) @ np.array([[1, 2], [3, 4], [5, 6]])`
(b) `np.array([[1, 2, 3]]) + np.array([[1], [2], [3]])`
(c) `A = np.zeros((3, 4)); A[:, 0] = np.array([[1, 2, 3]])`
(d) `np.ones((3,)) @ np.ones((5,))`

In [ ]:
# (a)
print('(a):', np.array([1, 2, 3]) @ np.array([[1, 2], [3, 4], [5, 6]]))

# (b)
print('(b):\n', np.array([[1, 2, 3]]) + np.array([[1], [2], [3]]))

# (c)
try:
    A = np.zeros((3, 4))
    A[:, 0] = np.array([[1, 2, 3]])
    print('(c): assigned OK (shape got squeezed)')
    print(A)
except ValueError as e:
    print('(c) error:', e)

# (d)
try:
    np.ones((3,)) @ np.ones((5,))
except ValueError as e:
    print('(d) error:', e)

<details>
<summary><b>Show solution</b></summary>

- (a) **Works**: `(3,) @ (3, 2) = (2,)` — vector-matrix product.
- (b) **Works**: `(1, 3) + (3, 1)` broadcasts to `(3, 3)` — classic outer-style broadcasting.
- (c) **Works** in NumPy — the assignment broadcasts `(1, 3)` down to the column shape `(3,)`. Technically a shape mismatch, but NumPy squeezes it silently.
- (d) **Error**: `(3,) @ (5,)` — inner dimensions must match for 1D-1D matmul (which computes the dot product); 3 ≠ 5.

</details>

### Q39 — Softmax
`scores = np.array([3.0, 1.0, 2.0, 5.0])`.
- (a) Compute softmax. Shape?
- (b) For `scores` shape `(32, 10)`, compute per-row softmax. What `axis` and `keepdims` do you need?

In [ ]:
def softmax(x, axis=None):
    x = x - x.max(axis=axis, keepdims=True)  # stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

scores = np.array([3.0, 1.0, 2.0, 5.0])
p = softmax(scores)
print('1D softmax:', p, '  sum:', p.sum())

batch = np.random.randn(32, 10)
pb = softmax(batch, axis=1)
print('\nbatched shape:', pb.shape)
print('per-row sums (should all be 1):', pb.sum(axis=1)[:5], '...')

<details>
<summary><b>Show solution</b></summary>

- (a) Shape `(4,)`. Trick: subtract `max` first for numerical stability.
- (b) Use `axis=1` (reduce across classes), `keepdims=True` so the normalization broadcasts back to `(32, 10)`.

A fully general implementation (see code cell) takes an `axis` parameter so it works for 1D or batched inputs.

</details>

### Q40 — Vector field composition
$f(x) = \sigma(Wx + b)$ with $W: (10, 3)$, $b: (10,)$, $x: (3,)$.
- (a) Shape of $f(x)$?
- (b) Vectorize: $X: (100, 3)$. Rewrite $f$ for the batch. Output shape?
- (c) Broadcasting trick for $b$?

In [ ]:
W = np.random.randn(10, 3)
b = np.random.randn(10)
x = np.random.randn(3)

# Single example
f_x = sigmoid(W @ x + b)
print('single:', f_x.shape)

# Batched (rows are examples)
X = np.random.randn(100, 3)
# Z = X @ W.T + b   →   (100, 3) @ (3, 10) + (10,) → (100, 10)
Z = X @ W.T + b
F = sigmoid(Z)
print('batched:', F.shape)

<details>
<summary><b>Show solution</b></summary>

- (a) `(10,)` — single output vector.
- (b) Batched: `Z = X @ W.T + b` gives shape `(100, 10)`. Each row of the output is one example's layer output. Alternatively, keeping the column-per-example convention: `Z = W @ X.T + b[:, None]` → `(10, 100)`.
- (c) `b` has shape `(10,)` and `X @ W.T` has shape `(100, 10)`. NumPy aligns from the right: `(100, 10) + (10,)` broadcasts cleanly across the 100 rows — one bias per output neuron added to each example.

</details>

## That's all 40 questions

Work through them at your own pace. Each code cell is scratch space — modify freely to deepen intuition. Click **Show solution** when you want to check your answer.

## Part B — Foundations

### Q16 — Dot product as similarity
- (a) Write $a \cdot b$ as an explicit sum.
- (b) Compute $\dfrac{\partial (a \cdot b)}{\partial a}$.
- (c) Why does a larger dot product mean "more similar"?

In [ ]:
# Explore: dot product grows with alignment
a = np.array([1.0, 0.0])
for angle_deg in [0, 30, 60, 90, 120, 180]:
    theta = np.deg2rad(angle_deg)
    b = np.array([np.cos(theta), np.sin(theta)])
    print(f'angle {angle_deg:>3}°   a·b = {a @ b:+.3f}')

<details>
<summary><b>Show solution</b></summary>

- (a) $a \cdot b = \sum_j a_j b_j$
- (b) $\dfrac{\partial (a \cdot b)}{\partial a} = b$. Each entry $a_j$ contributes $a_j b_j$, so $\partial/\partial a_j = b_j$. The gradient has the same shape as $a$ because there's one slope per entry.
- (c) $a \cdot b = \|a\| \|b\| \cos \theta$. For fixed magnitudes, larger dot product ⇔ smaller angle ⇔ vectors point in more similar directions.

</details>

### Q17 — Sigmoid identities
- (a) Range of $\sigma$?
- (b) Prove $\sigma(-x) = 1 - \sigma(x)$.
- (c) Prove $\sigma'(x) = \sigma(x)(1 - \sigma(x))$.

In [ ]:
x = np.linspace(-6, 6, 100)
s = sigmoid(x)
# Verify symmetry
print('sigmoid(-x) ≈ 1 - sigmoid(x)?', np.allclose(sigmoid(-x), 1 - s))
# Verify derivative identity via finite differences
dx = 1e-5
numeric_deriv = (sigmoid(x + dx) - sigmoid(x - dx)) / (2 * dx)
analytic_deriv = s * (1 - s)
print('derivative identity holds?    ', np.allclose(numeric_deriv, analytic_deriv))

<details>
<summary><b>Show solution</b></summary>

- (a) **(0, 1)**. As $x \to -\infty$, $\sigma \to 0$; as $x \to +\infty$, $\sigma \to 1$. Never exactly 0 or 1.
- (b) $1 - \sigma(x) = 1 - \dfrac{1}{1 + e^{-x}} = \dfrac{e^{-x}}{1 + e^{-x}} = \dfrac{1}{e^x + 1} = \sigma(-x)$. ✓
- (c) $\sigma(x) = (1 + e^{-x})^{-1}$. By chain rule:
$$\sigma'(x) = -(1 + e^{-x})^{-2} \cdot (-e^{-x}) = \frac{e^{-x}}{(1 + e^{-x})^2} = \sigma(x) \cdot \sigma(-x) = \sigma(x)(1 - \sigma(x)). \quad \blacksquare$$

</details>

### Q18 — From sigmoid to probability
In negative sampling:
- (a) $P(\text{real}) = ?$
- (b) $P(\text{noise}) = ?$
- (c) Show (b) equals $\sigma(-v \cdot v_i)$.

<details>
<summary><b>Show solution</b></summary>

- (a) $P(\text{real}) = \sigma(v \cdot v_i)$ — a sigmoid of the "score."
- (b) $P(\text{noise}) = 1 - \sigma(v \cdot v_i)$.
- (c) By the symmetry identity Q17(b): $1 - \sigma(v \cdot v_i) = \sigma(-v \cdot v_i)$. ✓

</details>

### Q19 — Likelihood to loss
Derive $J = -\log \sigma(v_o \cdot v_i) - \sum_n \log \sigma(-v_n \cdot v_i)$ from the likelihood of 1 real + $k$ noise pairs.

<details>
<summary><b>Show solution</b></summary>

Treat each pair as an independent Bernoulli. Joint likelihood:
$$\mathcal{L} = \sigma(v_o \cdot v_i) \cdot \prod_n \sigma(-v_n \cdot v_i).$$
Log-likelihood:
$$\log \mathcal{L} = \log \sigma(v_o \cdot v_i) + \sum_n \log \sigma(-v_n \cdot v_i).$$
Negate (we minimize negative log-likelihood):
$$J = -\log \sigma(v_o \cdot v_i) - \sum_n \log \sigma(-v_n \cdot v_i). \quad \blacksquare$$

This is binary cross-entropy: label 1 for the real pair, label 0 for each noise pair.

</details>

### Q20 — Log-sigmoid derivatives
- (a) Show $\dfrac{d}{dx}\log \sigma(x) = 1 - \sigma(x)$.
- (b) $\dfrac{d}{dx}[-\log \sigma(x)] = \sigma(x) - 1$.
- (c) $\dfrac{d}{dx}[-\log \sigma(-x)] = \sigma(x)$.

<details>
<summary><b>Show solution</b></summary>

Covered in Q9 — see those steps. Short version:
- (a) $\dfrac{\sigma'}{\sigma} = \dfrac{\sigma(1-\sigma)}{\sigma} = 1 - \sigma$.
- (b) Flip sign of (a).
- (c) Chain rule on inner $-x$: $-(1 - \sigma(-x)) = -\sigma(x)$. Negate: $\sigma(x)$.

</details>

### Q21 — Chain and sum rules
- (a) What is $\dfrac{\partial x}{\partial v_i}$ where $x = v \cdot v_i$?
- (b) Why does $\dfrac{\partial J}{\partial v_i}$ sum over multiple terms?
- (c) Write out $\dfrac{\partial J}{\partial v_i}$ fully.

<details>
<summary><b>Show solution</b></summary>

- (a) $\dfrac{\partial (v \cdot v_i)}{\partial v_i} = v$.
- (b) **Sum rule**: the derivative of a sum is the sum of derivatives. Since $J$ sums over the real pair + all negatives, each term contributes to the gradient on $v_i$.
- (c) $\dfrac{\partial J}{\partial v_i} = (\sigma(v_o \cdot v_i) - 1) v_o + \sum_n \sigma(v_n \cdot v_i) v_n$.

</details>

### Q22 — SGD update
$$\theta \leftarrow \theta - \eta \cdot \dfrac{\partial J}{\partial \theta}$$

- (a) What does $\eta$ control?
- (b) Why "stochastic"?
- (c) Which parameters get updated per step?

<details>
<summary><b>Show solution</b></summary>

- (a) $\eta$ is the **learning rate** — step size. Too large: unstable / diverging. Too small: slow / stuck.
- (b) "Stochastic" because gradients are computed on a *sample* (one pair + a few negatives), not the full dataset. The result is a noisy estimate of the true gradient, but computationally feasible on billions of pairs.
- (c) $v_i$ (target), $v_o$ (context), and each $v_n$ (negative sample). Everything else stays frozen for that step.

</details>

### Q23 — Putting it all together
Summarize word2vec negative sampling in a short paragraph.

<details>
<summary><b>Show solution</b></summary>

**Model**: predict whether a (word, context) pair is real using $P(\text{real}) = \sigma(v \cdot v_c)$ — logistic regression on the dot product of two embedding vectors.

**Loss**: for each real pair, draw $k$ random "negative" context words; minimize the binary cross-entropy that labels the real pair 1 and each negative 0. This gives $J = -\log \sigma(v_o \cdot v_i) - \sum_n \log \sigma(-v_n \cdot v_i)$.

**Learning**: stochastic gradient descent — one pair at a time, update $v_i$, $v_o$, and each $v_n$ using $\theta \leftarrow \theta - \eta \partial J / \partial \theta$.

**Key identities**: $\sigma'(x) = \sigma(x)(1 - \sigma(x))$ and $\sigma(-x) = 1 - \sigma(x)$. These make the gradients collapse to the clean form $(\sigma - 1) v$ for the real pair and $\sigma v$ for each negative — effectively 'prediction error times input'.

</details>

## Part C — Eigenvectors and the Connection to Word Embeddings

### Q24 — Definition
Define an eigenvector and eigenvalue of a square matrix $M$. State the equation and give a geometric interpretation.

In [ ]:
# Eigendecomposition of a small symmetric matrix
M = np.array([[4., 1.], [1., 3.]])
eigvals, eigvecs = np.linalg.eig(M)
print('eigenvalues:', eigvals)
print('eigenvectors (columns):\n', eigvecs)

# Verify: M @ v = λ v for each eigenpair
for i in range(len(eigvals)):
    lhs = M @ eigvecs[:, i]
    rhs = eigvals[i] * eigvecs[:, i]
    print(f'\nM @ v_{i} = {lhs}')
    print(f'λ_{i} * v_{i} = {rhs}')
    print(f'equal? {np.allclose(lhs, rhs)}')

<details>
<summary><b>Show solution</b></summary>

An **eigenvector** $v \ne 0$ of $M$ is a vector that satisfies
$$M v = \lambda v$$
for some scalar $\lambda$, the **eigenvalue**.

**Geometrically**: $M$ acts on $v$ by just **scaling** it (by $\lambda$) — no rotation, no change in direction. Eigenvectors are the "invariant axes" of the linear transformation $M$.

</details>

### Q25 — Eigenvectors vs singular vectors
- (a) Which SVD vectors, if any, are eigenvectors of some related matrix?
- (b) Show the columns of $V$ are eigenvectors of $A^\top A$. What are the eigenvalues?
- (c) Columns of $U$ are eigenvectors of what?

In [ ]:
# Verify: V columns are eigenvectors of A^T A
A = np.random.randn(6, 4)
U, S, Vt = np.linalg.svd(A, full_matrices=False)
V = Vt.T

AtA = A.T @ A
for j in range(4):
    v_j = V[:, j]
    lhs = AtA @ v_j
    rhs = (S[j]**2) * v_j
    print(f'σ_{j}^2 = {S[j]**2:.4f}   A^T A @ v_{j} matches σ^2 v?  {np.allclose(lhs, rhs)}')

<details>
<summary><b>Show solution</b></summary>

- (a) Both: columns of $V$ are eigenvectors of $A^\top A$; columns of $U$ are eigenvectors of $A A^\top$.

- (b) $A = U \Sigma V^\top \Rightarrow A^\top A = V \Sigma^\top U^\top U \Sigma V^\top = V \Sigma^2 V^\top$. So
$$A^\top A \cdot V = V \Sigma^2,$$
meaning each column $v_j$ satisfies $A^\top A \, v_j = \sigma_j^2 \, v_j$. **Eigenvalues are the squares of the singular values.**

- (c) Columns of $U$ are eigenvectors of $A A^\top$ (same eigenvalues $\sigma_j^2$).

</details>

### Q26 — Why eigenvectors matter for co-occurrence
Let $C = A^\top A$ (symmetric). Explain:
- (a) Why $C$ is symmetric and positive semi-definite.
- (b) Why the top eigenvectors of $C$ capture dominant data patterns.
- (c) The connection to using SVD on $A$ for word embeddings.

<details>
<summary><b>Show solution</b></summary>

- (a) **Symmetric**: $C^\top = (A^\top A)^\top = A^\top A = C$. **Positive semi-definite**: for any $x$, $x^\top C x = x^\top A^\top A x = (Ax)^\top (Ax) = \|Ax\|^2 \ge 0$.
- (b) Eigenvectors of $C$ correspond to directions of maximum variance (for centered data) or maximum "co-occurrence strength." The top one is the direction where projecting the data preserves the most variance; subsequent ones are orthogonal and capture decreasing amounts.
- (c) Doing SVD on $A$ *is* finding the eigenvectors of $A^\top A$ in disguise — same math, different packaging. The embedding $U_k \Sigma_k$ equivalently projects words onto the top eigenvectors of $A A^\top$ (the word-side similarity matrix). So word embeddings via SVD = projecting onto the dominant eigen-directions of the co-occurrence structure.

</details>

### Q27 — PCA connection
- (a) If you mean-center rows of $A$ first, how does SVD relate to PCA?
- (b) What are we "projecting onto" in both cases?

<details>
<summary><b>Show solution</b></summary>

- (a) PCA = eigendecomposition of the covariance matrix $\frac{1}{n-1}(A_c^\top A_c)$ where $A_c$ is mean-centered. SVD on $A_c$ gives $V$ as the eigenvectors of $A_c^\top A_c$ (up to scaling) — so SVD on mean-centered data is PCA.
- (b) Both project data onto the **top $k$ orthogonal directions of maximum variance**. For word co-occurrence, those directions represent the strongest "semantic axes" in the data.

</details>

### Q28 — Eigendecomposition vs SVD
- (a) Which matrices admit $M = Q \Lambda Q^{-1}$?
- (b) Which admit an SVD?
- (c) Why does SVD generalize eigendecomposition for rectangular matrices?

<details>
<summary><b>Show solution</b></summary>

- (a) **Diagonalizable** square matrices — not all squares are diagonalizable (e.g., Jordan-defective matrices). Symmetric real matrices are always diagonalizable with orthogonal $Q$.
- (b) **All matrices**, including rectangular ones. Every matrix has an SVD.
- (c) Rectangular matrices don't have eigenvectors in the standard sense ($Mv = \lambda v$ requires $M$ square). SVD provides the analogous "best axes" via left and right singular vectors, which generalize eigenvectors to any shape.

</details>

### Q29 — Power iteration
Describe how **power iteration** finds the top eigenvector:
$$v_{t+1} = \frac{M v_t}{\|M v_t\|}$$
Why does this converge? Why does it matter?

In [ ]:
# Power iteration for top eigenvector of a symmetric matrix
M = np.array([[4., 1., 2.],
              [1., 3., 0.],
              [2., 0., 5.]])

v = np.random.randn(3)
v /= np.linalg.norm(v)
for t in range(20):
    v = M @ v
    v /= np.linalg.norm(v)

# Compare with np.linalg.eigh (which sorts ascending)
eigvals, eigvecs = np.linalg.eigh(M)
true_top = eigvecs[:, -1]
print('power iteration result:', v)
print('true top eigenvector  :', true_top)
print('aligned (up to sign)? ', np.allclose(np.abs(v), np.abs(true_top), atol=1e-4))

<details>
<summary><b>Show solution</b></summary>

Write $v_0$ in the eigenvector basis: $v_0 = \sum_j c_j q_j$ where $M q_j = \lambda_j q_j$. Then:
$$M^t v_0 = \sum_j c_j \lambda_j^t q_j.$$
The term with the **largest $|\lambda_j|$** grows fastest, dominating all others. After normalization, $v_t$ aligns with the top eigenvector.

Normalizing each step prevents numerical overflow. Convergence rate depends on the ratio $|\lambda_2 / \lambda_1|$ — the smaller, the faster.

**Why it matters**: computing just one eigenvector in $O(\text{nnz})$ per iteration is far cheaper than full eigendecomposition. `svds` uses a related iterative method (Lanczos) to extract only the top $k$ singular vectors — essential for huge sparse matrices.

</details>

### Q30 — Eigenvalues and information content
- (a) What does "most informative" mean in terms of variance?
- (b) Why does throwing away small eigenvalues often **help** generalization?

<details>
<summary><b>Show solution</b></summary>

- (a) The $j$-th eigenvalue of $A^\top A$ is $\sigma_j^2$ — proportional to the **variance** of the data when projected onto $v_j$. The top $k$ directions preserve the most variance; equivalently, they give the smallest Frobenius reconstruction error. "Informative" ≈ "high variance" ≈ "contributes most to reconstructing $A$."
- (b) Small eigenvalues typically correspond to **noise** or idiosyncratic quirks of the training sample, not generalizable structure. Discarding them is a form of **regularization**: the compressed representation can't memorize noise, so it generalizes better. This is the same principle behind PCA denoising and the intuition for why deep models use bottleneck layers.

</details>

## Part D — Singular Value Decomposition

### Q1 — Factual recall
The Singular Value Decomposition factors a matrix $A$ of shape $(m, n)$ as $A = U \Sigma V^\top$. State the shape of each of $U$, $\Sigma$, and $V^\top$, and describe one property that $U$ and $V$ share.

In [ ]:
# Explore: factor a small matrix with SVD and verify shapes
A = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.],
              [1., 0., 1.]])
U, S, Vt = np.linalg.svd(A, full_matrices=True)
print('A :', A.shape)
print('U :', U.shape)
print('S :', S.shape, '   (just the diagonal, not the full matrix)')
print('Vt:', Vt.shape)

# Verify U and V have orthonormal columns
print('\nU.T @ U ≈ I?', np.allclose(U.T @ U, np.eye(U.shape[1])))
print('V.T @ V ≈ I?', np.allclose(Vt @ Vt.T, np.eye(Vt.shape[0])))

<details>
<summary><b>Show solution</b></summary>

- $U$ has shape $(m, m)$
- $\Sigma$ has shape $(m, n)$ — diagonal with singular values; `np.linalg.svd` returns just the diagonal as a 1D array of length $\min(m,n)$
- $V^\top$ has shape $(n, n)$

**Shared property**: both $U$ and $V$ have **orthonormal columns**, meaning $U^\top U = I$ and $V^\top V = I$. Geometrically, they're rotations/reflections of the space.

</details>

### Q2 — Interpretation
In the word co-occurrence setting, each row of $A$ corresponds to a word and each column to a context. After decomposition $A = U \Sigma V^\top$:

- (a) Which matrix contains "directions" that correspond to words?
- (b) What do the diagonal entries of $\Sigma$ represent intuitively?
- (c) Why are they sorted from largest to smallest?

<details>
<summary><b>Show solution</b></summary>

- (a) **$U$** — its rows correspond to words. Row $i$ of $U$ is word $i$'s coordinates in the "concept space."
- (b) Each $\sigma_j$ is the **importance / strength** of concept direction $j$. A large singular value means that direction accounts for a lot of the structure in $A$.
- (c) So that the truncation $A \approx U_k \Sigma_k V_k^\top$ **keeps the most informative directions**. If they were in arbitrary order, picking the top $k$ would be a worse approximation.

</details>

### Q3 — Truncated SVD
The truncated SVD approximates $A \approx U_k \Sigma_k V_k^\top$.

- (a) In what sense is this the "best" rank-$k$ approximation?
- (b) If the original co-occurrence rows are $V$-dimensional (vocab size), what is the embedding dimension?
- (c) Why does this give useful word embeddings?

In [ ]:
# Explore: reconstruction quality vs. k
from numpy.linalg import svd

np.random.seed(0)
A = np.random.randn(50, 30)
U, S, Vt = svd(A, full_matrices=False)

for k in [1, 5, 10, 20, 30]:
    Ak = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    err = np.linalg.norm(A - Ak)
    print(f'k={k:3d}  Frobenius error = {err:.3f}')

<details>
<summary><b>Show solution</b></summary>

- (a) It minimizes the **Frobenius norm** of the reconstruction error $\|A - A_k\|_F$ among all rank-$k$ matrices. This is the **Eckart–Young theorem**.
- (b) The embedding is $k$-dimensional (one row of $U_k$, optionally scaled by $\Sigma_k$).
- (c) Rows of $A$ that are similar (words appearing in similar contexts) project to similar rows of $U_k$. The distance in the low-dimensional embedding space reflects co-occurrence similarity, so near-synonyms end up nearby.

</details>

### Q4 — Shape computation
A co-occurrence matrix $A$ has shape $(10{,}000, 10{,}000)$. You run `svds(A, k=50)`.

- (a) Shape of $U$?
- (b) Shape of $S$?
- (c) Shape of $V^\top$?
- (d) Shape of `embeddings = U * S`?

In [ ]:
from scipy.sparse.linalg import svds
from scipy.sparse import random as sparse_random

A_sparse = sparse_random(10000, 10000, density=0.001, format='csr')
U, S, Vt = svds(A_sparse.astype(float), k=50)
print('U :', U.shape)
print('S :', S.shape)
print('Vt:', Vt.shape)
print('embeddings = U * S:', (U * S).shape)

<details>
<summary><b>Show solution</b></summary>

- (a) $U$ is **(10000, 50)**
- (b) $S$ is **(50,)** — just the diagonal
- (c) $V^\top$ is **(50, 10000)**
- (d) `embeddings = U * S` broadcasts the length-50 vector across the columns, giving **(10000, 50)**. Each row is a 50-dim word embedding.

</details>

### Q5 — Code gotcha
```python
U, S, Vt = svds(A.astype(float), k=50)
embeddings = U * S
```
What subtle issue does this code have? (Hint: compare `svds` to the usual convention.)

In [ ]:
# Demonstrate: svds returns singular values in ASCENDING order (opposite of np.linalg.svd)
from scipy.sparse.linalg import svds
from scipy.sparse import random as sparse_random

A_sparse = sparse_random(100, 100, density=0.05, format='csr').astype(float)
U, S, Vt = svds(A_sparse, k=10)
print('svds S:', S)   # ascending

# Fix: reverse to descending
U = U[:, ::-1]
S = S[::-1]
Vt = Vt[::-1, :]
print('reversed:', S)

<details>
<summary><b>Show solution</b></summary>

`svds` returns singular values in **ascending** order (smallest first), the opposite of the standard math convention and the opposite of `np.linalg.svd`. So `U[:, 0]` corresponds to the *least* important direction, not the most.

**Fix**: reverse all three arrays:
```python
U, S, Vt = U[:, ::-1], S[::-1], Vt[::-1, :]
```

</details>

### Q6 — Why not keep all singular values?
A classmate claims keeping all singular values gives a "better" embedding because no information is lost. Give at least two reasons the truncated version is actually preferred.

<details>
<summary><b>Show solution</b></summary>

1. **Noise suppression**: small singular values typically correspond to noise or idiosyncratic patterns. Keeping them bakes noise into the embedding, hurting generalization.
2. **Dimensionality**: a 50-dim vector is far more useful (fits in memory, fast to compare, works with downstream models) than a 50,000-dim sparse row.
3. **Semantic structure**: truncation forces the embedding to capture the *dominant* patterns — the ones that generalize. This is why truncated embeddings cluster by meaning.
4. **Computational cost**: full SVD on a 50k × 50k matrix is infeasible; only top-$k$ methods scale.

</details>

### Q7 — Why `svds` instead of `svd`?
`np.linalg.svd(A)` computes the full decomposition. Why use `scipy.sparse.linalg.svds(A, k=50)` for a 50,000 × 50,000 co-occurrence matrix instead?

<details>
<summary><b>Show solution</b></summary>

Three reasons:
1. **Sparse-aware**: `svds` works on sparse matrices (CSR/CSC) without densifying — a dense 50k × 50k float64 matrix would be 20 GB.
2. **Partial decomposition**: `svds` only computes the top $k$ singular values/vectors using iterative methods (related to power iteration / Lanczos). `np.linalg.svd` computes all 50,000 — overkill when you want 50.
3. **Memory and time**: full SVD is $O(mn \min(m,n))$ time and dense output; iterative top-$k$ is $O(k \cdot \text{nnz})$ per iteration.

</details>

## Part E — Word2Vec Negative Sampling

The negative sampling loss for one target $v_i$, context word $v_o$, and $k$ negatives $v_n$ is:

$$J = -\log \sigma(v_o \cdot v_i) - \sum_n \log \sigma(-v_n \cdot v_i)$$

In [ ]:
# Set up: sigmoid and a toy configuration
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

np.random.seed(1)
dim = 5
v_i = np.random.randn(dim)         # target
v_o = np.random.randn(dim)         # context (real)
v_negs = np.random.randn(3, dim)   # 3 negative samples

print('v_o · v_i =', v_o @ v_i)
print('v_n · v_i for each negative:', v_negs @ v_i)
print('sigmoid(v_o · v_i)  =', sigmoid(v_o @ v_i))
print('sigmoid(-v_n · v_i) =', sigmoid(-(v_negs @ v_i)))

### Q8 — Loss function reading

- (a) When is the first term small? What is it trying to accomplish?
- (b) When is the second term small? What is it trying to accomplish?
- (c) Why not just $\sigma(v_n \cdot v_i)$ in the second term?

<details>
<summary><b>Show solution</b></summary>

- (a) The first term $-\log \sigma(v_o \cdot v_i)$ is small when $v_o \cdot v_i$ is **large positive**, making $\sigma \approx 1$ and $-\log \sigma \approx 0$. It **pulls the real pair together**.
- (b) The second term is small when each $v_n \cdot v_i$ is **very negative**, making $\sigma(-v_n \cdot v_i) \approx 1$. It **pushes noise pairs apart**.
- (c) The negative sign makes it a probability of being *noise*. $P(\text{noise}) = 1 - \sigma(v \cdot v_i) = \sigma(-v \cdot v_i)$ (sigmoid symmetry). We want to maximize that probability for noise pairs → equivalently, minimize $-\log \sigma(-v_n \cdot v_i)$.

</details>

### Q9 — Sigmoid-log identities
Prove:

- (a) $\dfrac{d}{dx}\!\left[-\log \sigma(x)\right] = \sigma(x) - 1$
- (b) $\dfrac{d}{dx}\!\left[-\log \sigma(-x)\right] = \sigma(x)$

<details>
<summary><b>Show solution</b></summary>

Using $\sigma'(x) = \sigma(x)(1 - \sigma(x))$ and $\sigma(-x) = 1 - \sigma(x)$:

**(a)**
$$\frac{d}{dx}\log \sigma(x) = \frac{\sigma'(x)}{\sigma(x)} = \frac{\sigma(x)(1 - \sigma(x))}{\sigma(x)} = 1 - \sigma(x).$$
Negate:
$$\frac{d}{dx}[-\log \sigma(x)] = \sigma(x) - 1. \quad \blacksquare$$

**(b)** Chain rule on inner $-x$ (derivative $-1$):
$$\frac{d}{dx}\log \sigma(-x) = \frac{\sigma'(-x)}{\sigma(-x)} \cdot (-1) = -(1 - \sigma(-x)) = -\sigma(x).$$
(Used $1 - \sigma(-x) = \sigma(x)$.) Negate:
$$\frac{d}{dx}[-\log \sigma(-x)] = \sigma(x). \quad \blacksquare$$

</details>

### Q10 — Vector gradients
Using $\dfrac{\partial(a \cdot b)}{\partial a} = b$ and the identities from Q9, derive:

- (a) $\dfrac{\partial J}{\partial v_o}$
- (b) $\dfrac{\partial J}{\partial v_n}$ for one negative
- (c) $\dfrac{\partial J}{\partial v_i}$

In [ ]:
# Verify gradients numerically
def J(v_i, v_o, v_negs):
    return -np.log(sigmoid(v_o @ v_i)) - np.sum(np.log(sigmoid(-(v_negs @ v_i))))

def grad_v_o_analytic(v_i, v_o):
    return (sigmoid(v_o @ v_i) - 1) * v_i

def grad_v_o_numeric(v_i, v_o, v_negs, eps=1e-6):
    g = np.zeros_like(v_o)
    for j in range(len(v_o)):
        v_o_plus = v_o.copy(); v_o_plus[j] += eps
        v_o_minus = v_o.copy(); v_o_minus[j] -= eps
        g[j] = (J(v_i, v_o_plus, v_negs) - J(v_i, v_o_minus, v_negs)) / (2 * eps)
    return g

print('analytic:', grad_v_o_analytic(v_i, v_o))
print('numeric :', grad_v_o_numeric(v_i, v_o, v_negs))
print('match?   ', np.allclose(grad_v_o_analytic(v_i, v_o), grad_v_o_numeric(v_i, v_o, v_negs)))

<details>
<summary><b>Show solution</b></summary>

Chain rule: $J$ depends on $v_o$ only through $x = v_o \cdot v_i$, with $\partial x / \partial v_o = v_i$.

**(a)** From Q9(a): $\dfrac{\partial J}{\partial v_o} = (\sigma(v_o \cdot v_i) - 1) \cdot v_i$

**(b)** $J$ depends on $v_n$ only through $-v_n \cdot v_i$. By Q9(b) and chain rule: $\dfrac{\partial J}{\partial v_n} = \sigma(v_n \cdot v_i) \cdot v_i$

**(c)** $v_i$ appears in all terms:
$$\frac{\partial J}{\partial v_i} = (\sigma(v_o \cdot v_i) - 1) \cdot v_o + \sum_n \sigma(v_n \cdot v_i) \cdot v_n$$

</details>

### Q11 — Direction interpretation

$$\frac{\partial J}{\partial v_o} = (\sigma(v_o \cdot v_i) - 1)\, v_i$$

- (a) What is the sign of $\sigma(v_o \cdot v_i) - 1$? Why does this move $v_o$ toward $v_i$?
- (b) When $v_o \cdot v_i$ is already very large, what happens to the gradient magnitude?

<details>
<summary><b>Show solution</b></summary>

- (a) $\sigma \in (0, 1)$, so $\sigma - 1 \in (-1, 0)$ — **negative**. The gradient is a negative multiple of $v_i$. SGD subtracts the gradient, so $v_o \leftarrow v_o + |\sigma - 1| \cdot v_i$ — adding a bit of $v_i$ to $v_o$ moves it **toward** $v_i$.
- (b) When $v_o \cdot v_i$ is large, $\sigma \approx 1$, so $|\sigma - 1| \approx 0$ — tiny gradient. This is **diminishing returns**: once the model has learned the pair, it stops wasting updates on it.

</details>

### Q12 — Negative sample dynamics

$$\frac{\partial J}{\partial v_n} = \sigma(v_n \cdot v_i)\, v_i$$

- (a) Which direction does SGD move $v_n$?
- (b) What happens when $v_n$ is doing its job (score very negative)?
- (c) When is the gradient largest, and why does that make sense?

<details>
<summary><b>Show solution</b></summary>

- (a) $\sigma > 0$, so gradient is a positive multiple of $v_i$. SGD subtracts, moving $v_n$ **away from** $v_i$.
- (b) When $v_n \cdot v_i$ is very negative, $\sigma \approx 0$ — tiny gradient. The sample is already well-separated; no update needed.
- (c) Largest when $v_n \cdot v_i$ is positive and large — i.e., when the model *wrongly* thinks the noise pair is similar. Biggest correction goes to the biggest mistake. Exactly the right behavior.

</details>

### Q13 — The target vector gets both effects

$$\frac{\partial J}{\partial v_i} = (\sigma(v_o \cdot v_i) - 1)\, v_o + \sum_n \sigma(v_n \cdot v_i)\, v_n$$

- (a) What does each part do to $v_i$?
- (b) If $v_n \approx v_o$ (negative looks like context), what happens?

<details>
<summary><b>Show solution</b></summary>

- (a) The **first part** pulls $v_i$ toward $v_o$ (same logic as Q11). The **sum** pushes $v_i$ away from every $v_n$. $v_i$ ends up placed to be close to its real context and far from noise.
- (b) The two gradient contributions partially **cancel**: pulling toward $v_o$ and pushing from $v_n \approx v_o$ roughly offset. In practice this is rare because negatives are sampled uniformly or by unigram frequency — unlikely to collide with a specific context. The $\sigma$ weights also down-weight contradictory signals when the pair scores are already consistent.

</details>

### Q14 — Why negatives at all
Why does word2vec need the sum over negatives? What goes wrong without it?

<details>
<summary><b>Show solution</b></summary>

Without negatives, the loss is just $-\log \sigma(v_o \cdot v_i)$. The trivial minimum sets all vectors identical and large in some direction — then every $v \cdot v_i \to \infty$ and $\sigma \to 1$, so loss → 0. No learning signal to distinguish any words. Negatives force vectors to **spread out**: not all pairs can score high because negatives penalize that.

</details>

### Q15 — Connecting to BCE
Show that the negative sampling loss is a sum of binary cross-entropy terms: label 1 for $(v_o, v_i)$, label 0 for each $(v_n, v_i)$, with $p = \sigma(v \cdot v_i)$.

<details>
<summary><b>Show solution</b></summary>

BCE for a single example: $-y \log p - (1-y) \log(1-p)$.

- Real pair $(v_o, v_i)$ with label $y = 1$: $-\log \sigma(v_o \cdot v_i)$. ✓ (matches first term)
- Noise pair $(v_n, v_i)$ with label $y = 0$: $-\log(1 - \sigma(v_n \cdot v_i)) = -\log \sigma(-v_n \cdot v_i)$ (by sigmoid symmetry). ✓ (matches second term)

Sum: $-\log \sigma(v_o \cdot v_i) - \sum_n \log \sigma(-v_n \cdot v_i) = J$. Negative sampling is exactly $(k+1)$ independent BCE classifiers glued together.

</details>

## Part F — Word representation and sampling

Topics specific to word2vec's data pipeline: how words are encoded as vectors, and the sampling distribution used for picking negative samples.

### Q41 — Zipf's law
**Zipf's law** says that in natural language, the frequency of a word is roughly inversely proportional to its rank: the $r$-th most common word appears $\propto 1/r$ times.

- (a) If "the" (rank 1) appears 10,000 times, roughly how often does the 100th most common word appear?
- (b) Plot word frequency vs rank on a log-log plot. What shape do you expect?
- (c) Why do a few words account for most of the text?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate a Zipfian distribution over 10,000 words
V = 10000
ranks = np.arange(1, V + 1)
freqs = 10000 / ranks     # Zipf: freq ~ 1/rank

print(f'Rank 1   (e.g. "the"):   {freqs[0]:.0f}')
print(f'Rank 10:                 {freqs[9]:.0f}')
print(f'Rank 100:                {freqs[99]:.0f}')
print(f'Rank 1000:               {freqs[999]:.0f}')
print(f'Rank 10000:              {freqs[-1]:.0f}')

plt.figure(figsize=(6, 4))
plt.loglog(ranks, freqs)
plt.xlabel('rank (log)')
plt.ylabel('frequency (log)')
plt.title("Zipf's law: freq ∝ 1 / rank")
plt.grid(True, which='both', alpha=0.3)
plt.show()

<details>
<summary><b>Show solution</b></summary>

- (a) ~ **100 times** — frequency drops as $1/r$, so rank 100 is roughly 1/100 the frequency of rank 1.
- (b) A **nearly straight line with slope ≈ −1**. The log-log plot linearizes the power law $f = C/r$ into $\log f = \log C - \log r$.
- (c) The sharp drop-off means ~100 of the most common words typically account for ~50% of all tokens. This is why stopword lists exist (common words carry little info) and why rare words dominate the vocabulary count (long tail of hapax legomena).

</details>

### Q42 — The $0.75$ power trick
Word2vec samples negatives from a modified unigram distribution:
$$P(w) \propto f(w)^{0.75}$$
where $f(w)$ is the raw frequency.

- (a) What does raising to a fractional power do to the shape of the distribution?
- (b) Why is this better than sampling proportionally to $f(w)$ directly? Than uniformly?

In [ ]:
# Compare three sampling distributions over a Zipfian vocab
freqs = 10000 / np.arange(1, 101)   # top 100 words

p_uniform  = np.ones_like(freqs) / len(freqs)
p_linear   = freqs / freqs.sum()
p_075      = freqs**0.75 / (freqs**0.75).sum()

for name, p in [('uniform', p_uniform), ('f(w)', p_linear), ('f(w)^0.75', p_075)]:
    print(f'{name:12s}  P(rank 1): {p[0]:.4f}   P(rank 100): {p[-1]:.4f}   ratio: {p[0]/p[-1]:.1f}')

<details>
<summary><b>Show solution</b></summary>

- (a) Raising to a fractional power (0 < p < 1) **flattens** the distribution: it reduces the gap between the most frequent and least frequent words. $f^{0.75}$ is between $f$ (unchanged) and $f^0 = 1$ (uniform).

- (b) Trade-off between extremes:
  - **Pure $f(w)$**: nearly always samples "the", "of", "and". Negatives become redundant — they're too easy to distinguish from any specific context word. Model wastes updates on obvious noise.
  - **Uniform**: rare words get sampled too often. These rare words have poorly-trained embeddings, so they're also easy to distinguish. Also redundant.
  - **$f(w)^{0.75}$**: compromise — common words still appear often enough to be meaningful negatives, rare words appear often enough that they get training signal. Empirically found to work best (Mikolov et al., 2013).

</details>

### Q43 — One-hot vector
A **one-hot vector** encodes a single item from a discrete set.

- (a) For vocabulary `['cat', 'dog', 'fish', 'bird', 'frog']` (size 5), write the one-hot vector for `'fish'`.
- (b) Shape? What's special about its contents?
- (c) When would you use a one-hot encoding in word2vec?

In [ ]:
vocab = ['cat', 'dog', 'fish', 'bird', 'frog']
word2index = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

def one_hot(word, V, word2index):
    v = np.zeros(V, dtype=int)
    v[word2index[word]] = 1
    return v

for w in ['cat', 'fish', 'frog']:
    print(f'{w}: {one_hot(w, V, word2index)}')

<details>
<summary><b>Show solution</b></summary>

- (a) `'fish'` is index 2, so: `[0, 0, 1, 0, 0]`.
- (b) Shape `(V,)` where V is vocabulary size. Exactly **one** entry equals 1; the rest are 0.
- (c) Encoding a **single word** — the target word $w_i$ or the single positive context word $w_o$ in skip-gram.

**Note**: in practice, we almost never build the actual one-hot vector. Multiplying $W \cdot \text{one-hot}$ just picks out one column of $W$, so we use an index lookup directly (e.g., `W[:, idx]` or `W[idx]`) — same result, no wasted memory.

</details>

### Q44 — Multi-hot vector
A **multi-hot vector** encodes a *set* of items from a discrete vocabulary.

- (a) For the same vocabulary, write the multi-hot vector for `{'cat', 'frog'}`.
- (b) Shape? How is it different from one-hot?
- (c) Where does multi-hot appear in word2vec?

In [ ]:
def multi_hot(words, V, word2index):
    v = np.zeros(V, dtype=int)
    for w in words:
        v[word2index[w]] = 1
    return v

print('cat + frog:', multi_hot(['cat', 'frog'], V, word2index))
print('all:       ', multi_hot(vocab, V, word2index))
print('empty:     ', multi_hot([], V, word2index))

<details>
<summary><b>Show solution</b></summary>

- (a) `'cat'` is index 0, `'frog'` is index 4: `[1, 0, 0, 0, 1]`.
- (b) Same shape `(V,)`. **Multiple** entries equal 1 — one per selected word.
- (c) Encoding the **set of negative samples** $W_n$. If you draw $k$ noise words for a given training step, you can represent them as a single multi-hot vector of shape `(V,)` with $k$ ones.

As with one-hot, we typically use a list/array of indices instead of the full `(V,)` vector in practice.

</details>

### Q45 — From indices to vectors to indices
The docstring you were reading says:

```
Returns:
    - wi: target vector index
    - wo: context vector index
    - Wn: negative vectors index
```

- (a) The docstring says it returns **one-hot and multi-hot vectors**, but the names end in **"index"**. What's going on?
- (b) If `wi = 42`, how would you compute the actual target embedding from a weight matrix `W` of shape `(V, dim)`?
- (c) If `Wn = [17, 203, 45, 1]` (4 negative samples), how would you extract all 4 of their embeddings at once?

In [ ]:
# Suppose we have an embedding matrix with 100 words of dim 10
V, dim = 100, 10
W = np.random.randn(V, dim)

wi = 42
wo = 7
Wn = [17, 23, 45, 1]

# Target embedding (one row = one word's vector)
v_i = W[wi]
print('v_i shape:', v_i.shape)

# Context embedding (same lookup)
v_o = W[wo]
print('v_o shape:', v_o.shape)

# Negative embeddings — fancy indexing: gather multiple rows at once
V_n = W[Wn]
print('V_n shape:', V_n.shape, '  (4 negatives, each dim 10)')

<details>
<summary><b>Show solution</b></summary>

- (a) The docstring is conceptually describing one-hot/multi-hot vectors, but the implementation returns **indices** because that's dramatically more efficient. A one-hot vector of length 50,000 wastes 49,999 zeros; the single nonzero index is all you need. Modern code almost always passes indices around and does lookups.

- (b) `v_i = W[42]` — one row of the embedding matrix. Shape `(dim,)`. Mathematically equivalent to `one_hot(42) @ W` but no multiplication needed.

- (c) `V_n = W[[17, 203, 45, 1]]` — **fancy indexing** gathers rows at the listed positions in one shot. Shape `(4, dim)`. Mathematically equivalent to `multi_hot(...) @ W` after reshaping, but far faster and uses less memory.

</details>

### Q46 — Why index-based is faster
Suppose vocab size $V = 50{,}000$ and embedding dim $d = 300$. Compare the cost of:

- (a) Multiplying a one-hot vector of shape `(V,)` by a weight matrix of shape `(V, d)` to get the target embedding of shape `(d,)`.
- (b) Doing `W[idx]` to get the same thing.

Which is better and why?

In [ ]:
import time
V, d = 50_000, 300
W = np.random.randn(V, d)
idx = 42
oh = np.zeros(V); oh[idx] = 1

t0 = time.time()
for _ in range(1000):
    v = oh @ W            # full matmul
t_mul = time.time() - t0

t0 = time.time()
for _ in range(1000):
    v = W[idx]            # indexed lookup
t_idx = time.time() - t0

print(f'matmul   : {t_mul*1000:6.1f} ms  (over 1000 reps)')
print(f'lookup   : {t_idx*1000:6.1f} ms  (over 1000 reps)')
print(f'speedup  : {t_mul / t_idx:5.0f}x')

<details>
<summary><b>Show solution</b></summary>

Mathematically they produce the same result:
$$\text{one-hot}(i)^\top \cdot W = W[i, :]$$

- (a) Full matmul: $V \cdot d = 50{,}000 \times 300 = 15{,}000{,}000$ multiplications. Nearly all are $0 \cdot W_{jk}$, which contribute nothing.
- (b) Indexed lookup: **1** memory read for a $d$-dim slice. No arithmetic.

Lookup is typically ~1000× faster. This is why real word2vec implementations never build the one-hot vector — they just use `W[idx]`. The one-hot representation is conceptually useful for deriving the math, but the index-based implementation is the one you run.

</details>

### Q47 — Sampling negatives from a Zipfian
Write a function that draws $k$ negative samples from a Zipfian-like distribution, excluding the current positive pair.

In [ ]:
# Build sampling distribution: freqs^0.75, normalized
V = 1000
raw_freqs = 10000 / np.arange(1, V + 1)   # Zipfian
sampling_p = raw_freqs ** 0.75
sampling_p /= sampling_p.sum()

def sample_negatives(k, sampling_p, exclude):
    """Draw k negatives, re-sampling if we hit any excluded index."""
    exclude = set(exclude)
    negatives = []
    while len(negatives) < k:
        candidates = np.random.choice(len(sampling_p), size=k, p=sampling_p)
        for c in candidates:
            if c not in exclude:
                negatives.append(int(c))
                if len(negatives) == k:
                    break
    return negatives

# Example: target word index 42, context 7, draw 5 negatives
negs = sample_negatives(5, sampling_p, exclude={42, 7})
print('Negatives:', negs)

# Distribution check: ranks should skew lower (more common words), with 0.75 flattening
counts = np.bincount([n for _ in range(10000) for n in sample_negatives(1, sampling_p, exclude=set())], minlength=V)
print('\nTop 5 most-sampled indices (should be low = common words):', np.argsort(-counts)[:5])

<details>
<summary><b>Show solution</b></summary>

Key points about this function:
1. Pre-compute the sampling distribution once (not per call) — building and normalizing a `(V,)` probability vector is expensive.
2. Use `np.random.choice(V, size=k, p=p)` for weighted sampling.
3. Loop + exclude check handles the (rare) case of drawing the positive pair as a negative. In practice this is so uncommon that some implementations skip the check for speed.
4. The $f^{0.75}$ weighting ensures common words show up often but not overwhelmingly, and rare words still get sampled.

For *huge* vocabularies, there's an even faster technique: **unigram table** (an array of ~1e8 ints where each word appears with frequency proportional to $p$). Sampling is just a random array index — O(1), no normalization needed per call.

</details>

## Part G — Skip-Gram Word2Vec: Concept Walkthrough

A conceptual tour of the 9 ideas that make word2vec work, ordered from foundations to the high-level theoretical insight. Each question builds on the previous ones.

1. **Distributional hypothesis** — the semantic premise
2. **Dot product as similarity score** — the math tool
3. **Skip-gram task framing** — what we're predicting
4. **Logistic loss (BCE)** — how we score predictions
5. **Negative sampling** — a scalable alternative to softmax
6. **Gradients via the chain rule** — how we compute updates
7. **SGD** — how we apply updates
8. **Two embedding matrices** — architectural choice
9. **Implicit matrix factorization** — the theoretical punchline

### Q48 — Distributional hypothesis
*"You shall know a word by the company it keeps."* — J. R. Firth, 1957.

- (a) State the distributional hypothesis in one sentence.
- (b) Why is this hypothesis the **justification** for the whole word2vec approach? What would collapse if it weren't true?
- (c) Give two example word pairs that illustrate the hypothesis and one that pushes against it.

<details>
<summary><b>Show solution</b></summary>

- (a) **Words that appear in similar contexts have similar meanings.**
- (b) Word2vec learns vectors by predicting the context around each word. If the distributional hypothesis didn't hold, context would carry no reliable semantic information — and no training signal would push semantically similar words to similar vectors. The entire premise of learning meaning from co-occurrence depends on this hypothesis.
- (c) Examples that support it:
  - *"doctor" / "physician"* — appear in similar sentences about medicine, hospitals, patients.
  - *"happy" / "joyful"* — appear in similar emotional contexts.

  Pushes against it: *"bank" (financial) / "bank" (river)* — homonyms with identical surface form but completely different contexts. One embedding can't cleanly represent both meanings. This is a known limitation that modern contextual models (BERT, GPT) address.

</details>

### Q49 — Dot product as a similarity score
In word2vec, the "score" assigned to a (target, context) pair is $v_o \cdot v_i$.

- (a) Why is the dot product a reasonable score? What does a large value mean geometrically?
- (b) When is $v_o \cdot v_i$ exactly zero, and what does that mean semantically?
- (c) Why do we *not* use the Euclidean distance $\|v_o - v_i\|$ as the score instead?

In [ ]:
import numpy as np

def sigmoid(x): return 1 / (1 + np.exp(-x))

# Three unit vectors at increasing angles
v_i = np.array([1.0, 0.0])
scores = {}
for angle in [0, 30, 60, 90, 120, 180]:
    theta = np.deg2rad(angle)
    v = np.array([np.cos(theta), np.sin(theta)])
    scores[angle] = v_i @ v
    print(f'angle {angle:>3}°    dot = {v_i @ v:+.3f}    sigmoid(dot) = {sigmoid(v_i @ v):.3f}')

<details>
<summary><b>Show solution</b></summary>

- (a) $v_o \cdot v_i = \|v_o\| \|v_i\| \cos\theta$. A larger value means the vectors point in more similar directions. Geometrically, large positive dot product ⇔ small angle ⇔ "same direction" ⇔ "similar words."
- (b) Zero dot product means the vectors are **orthogonal** (90° apart). Semantically, this corresponds to "unrelated" — the model has learned that one word's embedding tells you nothing about the other's.
- (c) Two reasons:
  1. **Scale invariance**: dot product directly rewards magnitude, which the model can exploit during training. Euclidean distance doesn't distinguish between "far because different direction" and "far because longer vector."
  2. **Gradient simplicity**: $\partial(v_o \cdot v_i)/\partial v_o = v_i$ — a single vector. Euclidean distance derivatives involve square roots and are messier.

  Also, dot product composes cleanly with the sigmoid: $\sigma(v_o \cdot v_i)$ naturally maps "similarity score" to "probability of being a real pair.

</details>

### Q50 — Skip-gram task framing
Skip-gram turns learning word embeddings into a **prediction task**: given a target word, predict words in its context window.

- (a) What is a "context window," and what parameter controls its size?
- (b) For the sentence `"the quick brown fox jumps"` with window size 2, what (target, context) training pairs does skip-gram generate when "brown" is the target?
- (c) Why does this prediction task produce meaningful vectors? Connect back to Q48.

In [ ]:
tokens = ['the', 'quick', 'brown', 'fox', 'jumps']
window = 2
target_idx = tokens.index('brown')

pairs = []
for j in range(max(0, target_idx - window), min(len(tokens), target_idx + window + 1)):
    if j != target_idx:
        pairs.append((tokens[target_idx], tokens[j]))

print('Skip-gram pairs for target "brown" (window=2):')
for p in pairs:
    print(' ', p)

<details>
<summary><b>Show solution</b></summary>

- (a) A context window is a span of positions around the target word (say, 2 to the left and 2 to the right). The window size is a hyperparameter; typical values are 2–10. Larger windows capture topical similarity; smaller windows capture syntactic similarity.
- (b) For "brown" (center), window = 2: `('brown', 'the')`, `('brown', 'quick')`, `('brown', 'fox')`, `('brown', 'jumps')`.
- (c) By the distributional hypothesis (Q48), words that appear in similar contexts have similar meanings. Skip-gram's training objective directly incentivizes the model to assign high scores to real (target, context) pairs. Words that frequently appear with similar contexts end up with similar target vectors because they both need to score well against the same set of context words.

</details>

### Q51 — Logistic loss (binary cross-entropy)
The dot product gives a raw score; the sigmoid turns it into a probability of being a real pair:
$$P(\text{real}) = \sigma(v_o \cdot v_i)$$

- (a) Why do we minimize $-\log \sigma(v_o \cdot v_i)$ instead of maximizing $\sigma(v_o \cdot v_i)$ directly?
- (b) Why does the $-\log$ amplify confident mistakes compared to just maximizing probability?
- (c) Connect BCE to the distributional-hypothesis story: what does minimizing BCE on skip-gram pairs force the model to learn?

In [ ]:
p = np.linspace(0.01, 0.99, 50)
loss = -np.log(p)
print(f'When model is confidently correct (p=0.99):  loss = {-np.log(0.99):.3f}')
print(f'When model is somewhat sure       (p=0.70):  loss = {-np.log(0.70):.3f}')
print(f'When model is unsure              (p=0.50):  loss = {-np.log(0.50):.3f}')
print(f'When model is confidently wrong   (p=0.01):  loss = {-np.log(0.01):.3f}')

<details>
<summary><b>Show solution</b></summary>

- (a) By convention, optimizers minimize. Negating the log-likelihood turns a maximization problem into a minimization. $-\log$ also has nicer numerical properties: products of probabilities become sums, avoiding floating-point underflow.
- (b) $-\log p$ grows **unboundedly** as $p \to 0$. Confidently wrong predictions ($p$ near 0 when the true label is 1) get huge loss, pushing the model to never be confidently wrong. Maximizing $p$ directly treats $p = 0.01$ (horrible) and $p = 0.49$ (bad) almost the same — both are small — so there's no strong gradient to escape $p = 0.01$.
- (c) It forces: for every observed (target, context) pair, make $v_o \cdot v_i$ large. Over the whole corpus, this means target vectors must point in directions where their real contexts live. Words with similar contexts end up pointing to similar context clusters → their target vectors align → they become similar embeddings.

</details>

### Q52 — Negative sampling as contrastive learning
Skip-gram's original softmax over the whole vocab $V$ costs $O(V)$ dot products per step — infeasible for $V = 100{,}000+$. **Negative sampling** replaces this with: real pair vs. $k$ random pairs.

- (a) What is the per-step cost of negative sampling (in dot products)?
- (b) Why does this count as "contrastive learning"?
- (c) Where do negative samples come from, and why that specific distribution (unigram raised to $0.75$)?
- (d) What would go wrong if we used only real pairs (no negatives)?

In [ ]:
V = 50_000
k = 5
print(f'Full softmax cost:     {V} dot products per step')
print(f'Negative sampling cost: {k + 1} dot products per step')
print(f'Speedup factor:         {V // (k + 1):,}x')

<details>
<summary><b>Show solution</b></summary>

- (a) **$k + 1$ dot products** — one for the real pair, $k$ for the negatives. Independent of vocabulary size. In practice $k = 5$–$20$.
- (b) Contrastive learning = learn by comparing a positive example to several negative examples. The model doesn't learn to predict the "right word" from thousands of options — it learns to **tell real pairs apart from random ones**. Much simpler binary signal, same effect on embedding geometry.
- (c) Negatives are drawn from $P(w) \propto f(w)^{0.75}$, where $f$ is unigram frequency. The $0.75$ power **flattens** the Zipfian distribution: ultra-common words (which would otherwise dominate) get down-weighted; rare words get seen often enough to train. (See Q42.)
- (d) Without negatives, the trivial minimum sets every vector in the same direction with huge magnitude — then every $v_o \cdot v_i$ is enormous, every $\sigma \to 1$, and loss $\to 0$. No learning signal distinguishes between words. Negatives force vectors to **spread out**: not all pairs can score high because negatives are penalized.

</details>

### Q53 — Gradients via the chain rule
The identity $\sigma'(x) = \sigma(x)(1 - \sigma(x))$ makes the gradients beautifully clean.

- (a) Walk through the chain-rule derivation of $\partial J / \partial v_o$ in one line (see Q10 for the full version).
- (b) Interpret the sign of each gradient: which direction does SGD move $v_o$, $v_n$, and $v_i$?
- (c) Why do the gradient magnitudes shrink as the model learns? What does this tell you about where learning concentrates?

In [ ]:
# Example gradients
np.random.seed(0)
d = 5
v_i = np.random.randn(d)
v_o = np.random.randn(d)
V_n = np.random.randn(4, d)

so = sigmoid(v_o @ v_i)
sn = sigmoid(V_n @ v_i)

grad_vo = (so - 1) * v_i
grad_vi = (so - 1) * v_o + sn @ V_n
grad_Vn = np.outer(sn, v_i)

print('grad v_o (should pull v_o toward v_i):', grad_vo)
print('|grad v_o|:', np.linalg.norm(grad_vo))
print('|grad v_i|:', np.linalg.norm(grad_vi))
print('|grad V_n| per row:', np.linalg.norm(grad_Vn, axis=1))

<details>
<summary><b>Show solution</b></summary>

- (a) $\dfrac{\partial J}{\partial v_o} = \dfrac{d(-\log\sigma(v_o \cdot v_i))}{d(v_o \cdot v_i)} \cdot \dfrac{\partial(v_o \cdot v_i)}{\partial v_o} = (\sigma(v_o \cdot v_i) - 1) \cdot v_i$.
- (b)
  - $v_o$: gradient is $(\sigma - 1) v_i$. Since $\sigma - 1 < 0$, SGD subtracts a negative multiple of $v_i$ — equivalently *adds* $v_i$ — moving $v_o$ **toward** $v_i$. **Pull.**
  - $v_n$: gradient is $\sigma v_i$. SGD subtracts a positive multiple of $v_i$, moving $v_n$ **away from** $v_i$. **Push.**
  - $v_i$: gets both — pulled toward $v_o$ and pushed away from every $v_n$.
- (c) When the model already scores a pair correctly ($\sigma \approx 1$ for real pairs, $\sigma \approx 0$ for noise), the gradient magnitude shrinks to zero. Learning automatically **concentrates on the mistakes**: the biggest updates happen on pairs the model got wrong. This is a form of built-in importance weighting.

</details>

### Q54 — Stochastic gradient descent (SGD)
SGD processes one training pair at a time, updating only the rows that were touched.

- (a) How many vectors get updated in a single SGD step (target, context, $k$ negatives)?
- (b) Why is updating only touched rows essential at word2vec scale?
- (c) Why do we use a smaller learning rate for SGD than for full-batch GD?

<details>
<summary><b>Show solution</b></summary>

- (a) **$k + 2$ vectors**: one target ($v_i$), one context ($v_o$), and $k$ negatives ($v_n$). All other rows of the embedding matrices are untouched — not even looked at.
- (b) Vocabulary could be 100k–1M+. Full-batch gradient descent would require computing gradients on every (word, context) pair in the corpus to make one update — infeasible for billion-word corpora. SGD lets you take a step after every single pair, so over billions of pairs you take billions of tiny steps — fast progress even without visiting the whole dataset per step.
- (c) Each SGD step uses a **noisy** gradient estimate from a single sample. A large learning rate amplifies that noise, causing the parameters to bounce around rather than converge. Smaller learning rate = smaller steps that smooth out the noise. In word2vec, typical $\eta$ is $0.025$ at start, linearly decayed to near zero.

</details>

### Q55 — Two embedding matrices
Word2vec maintains two separate embeddings per word: a **target** vector (when used as $v_i$) and a **context** vector (when used as $v_o$ or $v_n$).

- (a) What architectural problem does using two matrices solve?
- (b) What is the "$\|v\|^2$ self-similarity problem" that would arise with a single matrix?
- (c) After training, which matrix do practitioners typically keep as "the" embeddings, and why?

<details>
<summary><b>Show solution</b></summary>

- (a) It lets a word's role as "target" differ from its role as "context." A word like "the" is a common context for many other words, so its context vector may need to live in a central region. But as a target, "the" wants its own distinct position.

- (b) If a single matrix is used, then (word, word) self-pairs score $v \cdot v = \|v\|^2$. The model gets a strong signal to make $\|v\|^2$ large for every word just to maximize this trivial self-similarity. All vectors blow up in magnitude without learning useful geometry. Separate matrices avoid this: target vectors never dot with themselves.

- (c) The **target matrix** $V_i$ (also called "input" or "$W$" in some notations) is what people use downstream. The context matrix $V_o$ is often discarded. Some systems average or sum $V_i + V_o$ for slightly better quality.

In the Levy-Goldberg view, both are learned, but only $V_i$ is the "embedding" in the semantic sense.

</details>

### Q56 — Implicit matrix factorization
Levy & Goldberg (2014) showed that word2vec with negative sampling implicitly factorizes a specific co-occurrence matrix.

- (a) What matrix does word2vec with negative sampling implicitly approximate as $V_i \cdot V_o^\top$?
- (b) How does this connect word2vec to classical dimensionality reduction (SVD/PCA)?
- (c) If this is true, why do we train word2vec iteratively instead of just computing an SVD on the matrix?

<details>
<summary><b>Show solution</b></summary>

- (a) The **shifted Pointwise Mutual Information (PMI) matrix**:
$$M_{ij} = \log \frac{P(w_i, c_j)}{P(w_i)P(c_j)} - \log k$$
where $k$ is the number of negative samples. Word2vec with negative sampling learns $V_i V_o^\top \approx M$.

- (b) It means word embeddings can be computed in **two mathematically equivalent ways**:
  1. **Iterative SGD** (word2vec): sample pairs, apply gradient steps on $J$.
  2. **Explicit SVD**: build $M$ (shifted PMI), run SVD, take top $k$ components.

  Both approximate the same underlying matrix structure. SVD gives the exact best rank-$k$ approximation; word2vec gives an approximate one through SGD.

- (c) Three practical reasons:
  1. **Scale**: for 1M-word vocabulary, $M$ has $10^{12}$ entries. Building it explicitly needs terabytes. Word2vec never materializes $M$ — it samples one entry at a time.
  2. **Sparsity**: most of $M$ is zero or infinite ($\log 0$). SGD handles this gracefully; SVD doesn't.
  3. **Flexibility**: word2vec can learn from a stream, add new data, drop old data — SVD is a batch computation that has to redo everything.

  This insight unified two threads of NLP research — Mikolov's neural approach and the older "count-based" matrix-factorization approach are the same algorithm in different clothing.

</details>

### Q57 — Putting it all together
In 3–5 sentences, explain how **two words that share many contexts** end up with similar embeddings — tracing the chain through the distributional hypothesis, dot-product scoring, skip-gram pairs, BCE loss, negative sampling, chain-rule gradients, SGD, and two embedding matrices.

<details>
<summary><b>Show solution</b></summary>

Two words $w_1$ and $w_2$ that share many contexts (e.g., "doctor" and "physician") appear as the target in many skip-gram pairs with the same context words $c$. For each such pair, SGD applies $v_{w_1} \leftarrow v_{w_1} + \eta (1 - \sigma(v_c \cdot v_{w_1})) v_c$ — pulling $v_{w_1}$ toward $v_c$. Over billions of training steps, both $v_{w_1}$ and $v_{w_2}$ get pulled toward the **same set of context vectors** $v_c$, so they end up near each other in embedding space. Simultaneously, negative sampling pushes both of them away from random noise words, which (statistically) are not shared between $w_1$ and $w_2$ in any systematic way, so these pushes don't separate them. The emergent effect: shared contexts → shared pulls → similar target vectors → **semantic similarity encoded as geometric proximity**. This is exactly the geometry Levy & Goldberg showed is a factorization of the shifted PMI matrix.

</details>

## Part H — Loss Curves and the Zipfian Distribution

Reading, plotting, and interpreting training loss curves; deeper analysis of why natural-language word frequencies follow a power law and what that means for embedding models.

### Q58 — Expected shape of a healthy loss curve
When you plot training loss `Ls` vs. iteration during word2vec training, what does a **healthy** curve look like? Sketch its characteristic shape and describe each phase.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate a healthy loss curve: steep drop, then slow decay, with mini-batch noise
np.random.seed(0)
iters = 10000
clean = 2.5 * np.exp(-np.arange(iters) / 1500) + 0.3
noisy = clean + np.random.randn(iters) * 0.15

plt.figure(figsize=(8, 4))
plt.plot(noisy, alpha=0.3, label='raw per-step loss')
plt.plot(clean, color='red', label='underlying trend')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('Healthy training loss curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

<details>
<summary><b>Show solution</b></summary>

A healthy curve has three visible phases:

1. **Steep initial drop** — early iterations have huge gradients because weights are random and predictions are near 0.5 for every pair. Each step makes big corrections.
2. **Gradual flattening** — as embeddings start capturing structure, per-pair losses shrink. Gradient magnitudes decay by the same factor, so updates become smaller.
3. **Noisy plateau** — once the model is near a minimum, further updates mostly cancel out. The curve wobbles around a stable value but doesn't drop much more.

Overlaid: significant per-batch noise (because SGD uses one sample at a time) producing a thick band around the trend. Noise amplitude roughly constant throughout — what shrinks is the mean, not the variance.

</details>

### Q59 — Smoothing the loss curve
Raw per-step loss is spiky because each gradient uses one random sample. How do you compute a **moving average** to see the trend clearly?

- (a) Implement a simple rolling-mean smoother for a window size of your choice.
- (b) What's the trade-off in choosing the window size?

In [ ]:
def rolling_mean(x, window):
    cumsum = np.cumsum(np.insert(x, 0, 0.0))
    return (cumsum[window:] - cumsum[:-window]) / window

for w in [50, 200, 1000]:
    smooth = rolling_mean(noisy, w)
    plt.plot(np.arange(len(smooth)) + w // 2, smooth, label=f'window={w}')
plt.plot(noisy, alpha=0.1, color='gray', label='raw')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('Effect of smoothing window')
plt.legend()
plt.show()

<details>
<summary><b>Show solution</b></summary>

- (a) The cumulative-sum trick computes rolling means in $O(N)$ (versus the naive $O(Nk)$ nested-loop version). `np.convolve(x, np.ones(w)/w, mode='valid')` also works and is more readable.

- (b) **Trade-off**:
  - Small window → still spiky, but tracks real transitions quickly (e.g., learning-rate decay).
  - Large window → very smooth, but **lags** behind the true current loss and smears out sharp drops.

  Common choices: window = $\sqrt{N_{\text{iters}}}$ as a rough default. For 100k iterations, that's ~316. For loss curves, a window of 100–1000 is typical. Also consider plotting both the raw loss (translucent, `alpha=0.2`) and the smoothed one so you can see noise and trend together.

</details>

### Q60 — Diagnosing unhealthy curves
Match each loss-curve pattern to its likely cause and prescription.

Patterns:
1. Flat line, never decreases.
2. Starts dropping, then shoots up to infinity (or NaN).
3. Decreases, then suddenly explodes mid-training.
4. Decreases but very slowly — plateaus at a high value.
5. Decreases to a reasonable value, then starts increasing again.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
titles = ['1. flat', '2. shoots up', '3. explodes mid-training',
          '4. slow / stuck', '5. decreases then rises', 'healthy (reference)']

axes[0, 0].plot(np.ones(100) * 1.5 + np.random.randn(100) * 0.05)
axes[0, 1].plot(np.concatenate([np.exp(-np.linspace(0, 2, 30)) + 0.3,
                                 np.exp(np.linspace(0, 3, 70))]))
axes[0, 2].plot(np.concatenate([np.exp(-np.linspace(0, 2, 60)) + 0.3,
                                 np.concatenate([[2, 5, 10], 20 * np.ones(37)])]))
axes[1, 0].plot(2.5 - 0.5 * np.log(np.arange(1, 101)) / np.log(100) + np.random.randn(100) * 0.03)
axes[1, 1].plot(np.concatenate([2.5 * np.exp(-np.linspace(0, 2, 50)) + 0.3,
                                 0.3 + np.linspace(0, 1, 50)]))
axes[1, 2].plot(2.5 * np.exp(-np.arange(100) / 30) + 0.3 + np.random.randn(100) * 0.05)
for ax, t in zip(axes.flat, titles):
    ax.set_title(t); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<details>
<summary><b>Show solution</b></summary>

| Pattern | Likely cause | Prescription |
|---|---|---|
| 1. **Flat line** | Learning rate too low; no gradient flow; bug (e.g., zero gradients); dead sigmoid saturation | Increase lr; check that gradients are nonzero; check initialization scale |
| 2. **Shoots up** | Learning rate way too high; gradient explosion; init too large | Lower lr by 10×; shrink init scale |
| 3. **Explodes mid-training** | Numerical instability from a specific pair; gradient blow-up as weights grow; one bad batch | Lower lr; clip gradients; stabilize sigmoid (prevent $e^{-x}$ overflow) |
| 4. **Slow / stuck** | Learning rate too low; batch too small; bad init; saturated activations | Increase lr; investigate init; check feature scales |
| 5. **Decreases then rises** | Overfitting on a small train set; learning-rate schedule misconfigured; or SGD + no decay causing stepping over minimum | Smaller lr or add decay; early-stop; use more data |

</details>

### Q61 — Why the per-sample loss is so noisy
In word2vec with batch size 1 (pure SGD) and $k = 5$ negatives, each step evaluates one BCE for the real pair plus five for the negatives.

- (a) Why does the per-step loss vary wildly even when the model is well-trained?
- (b) Does the moving-average loss keep decreasing as training continues, or stabilize? Why?

<details>
<summary><b>Show solution</b></summary>

- (a) Even a well-trained model doesn't score every pair perfectly. Some real pairs — rare collocations, polysemous words — stay hard. If your current step happens to use one of those, the loss spikes. Also, negatives are random, and sometimes you draw a word that genuinely does collocate with the target (noise that isn't noise). The expected loss is low but the per-step variance remains high.
- (b) **Stabilizes.** The moving average converges to a floor (determined by the model's irreducible error on the data), not to zero. If you see it still dropping, training isn't done. If it's been flat for many epochs, you can stop. If it starts rising, you're overfitting or the lr has grown too aggressive relative to the current minimum.

</details>

### Q62 — Zipf's law: the equation
Zipf's law states that in natural language, word frequency is inversely proportional to rank:
$$f(w_r) = \frac{C}{r^s}$$
where $r$ is the rank, $s$ is the Zipf exponent (usually ≈ 1 for natural text), and $C$ is a normalization.

- (a) What does $s = 1$ imply about how fast frequencies drop with rank?
- (b) What would $s = 2$ look like? $s = 0.5$?
- (c) Why do we take $\log f$ vs $\log r$ when visualizing Zipfian data?

In [ ]:
V = 1000
ranks = np.arange(1, V + 1)
for s in [0.5, 1.0, 1.5, 2.0]:
    f = 1 / ranks**s
    plt.loglog(ranks, f, label=f's={s}')
plt.xlabel('rank (log)')
plt.ylabel('frequency (log)')
plt.title('Zipfian with varying exponent')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.show()

<details>
<summary><b>Show solution</b></summary>

- (a) $s = 1$: frequencies drop as $1/r$. The 10th word is 1/10 as frequent as the 1st; the 100th is 1/100. This is a **slow, heavy-tailed decay** — rare words are rare but not negligible in aggregate.
- (b) $s = 2$: much steeper. The 10th word is 1/100 as frequent as the 1st; rare words vanish. $s = 0.5$: much flatter. The 10th word is only $1/\sqrt{10} \approx 0.32$ as frequent — vocabulary is more "even."
- (c) Taking log of both sides of $f = C r^{-s}$:
$$\log f = \log C - s \log r.$$
A log-log plot linearizes the power law into a straight line with slope $-s$. You can **estimate** the Zipf exponent of your corpus by fitting a line to the log-log plot and reading off the negative slope.

</details>

### Q63 — Cumulative distribution of Zipfian text
In a Zipfian corpus, what fraction of total tokens is covered by the top $N$ most frequent words?

- (a) Compute and plot the **cumulative** share of tokens accounted for by the top 1, 10, 100, 1000 words.
- (b) In typical English text, the top ~150 words cover roughly 50% of all tokens. Why is this?
- (c) What practical implication does this have for a stopword list?

In [ ]:
V = 10_000
freqs = 1 / np.arange(1, V + 1)   # Zipf s=1
freqs /= freqs.sum()
cumulative = np.cumsum(freqs)

for n in [1, 10, 100, 500, 1000, 5000, 10000]:
    print(f'Top {n:>5} words cover  {cumulative[n-1]*100:5.1f}% of tokens')

plt.plot(cumulative)
plt.xscale('log')
plt.xlabel('rank (log)')
plt.ylabel('cumulative share of tokens')
plt.title("Zipfian: cumulative token share by rank")
plt.grid(True, alpha=0.3)
plt.show()

<details>
<summary><b>Show solution</b></summary>

- (a) Code output shows the dramatic head-vs-tail: top 10 words → ~30% of tokens, top 100 → ~50%, top 1000 → ~70%. The tail (90% of unique words) carries only ~30% of tokens.
- (b) Natural language has a small set of **function words** ("the", "of", "and", "to", "a") that appear in nearly every sentence. These high-frequency, low-semantic-content words dominate counts.
- (c) **Stopword lists are small and powerful**. Removing just the top ~150 function words cuts your dataset size in half without discarding much meaning. Many classical NLP pipelines (TF-IDF, LDA, bag-of-words classifiers) do this routinely.

For word2vec, stopwords aren't usually removed because their contexts are informative, but they are **subsampled** — a technique where frequent words are randomly dropped from training with probability $1 - \sqrt{t / f(w)}$ for a threshold $t$. This gives both benefits: lower training cost and better embeddings for rare words.

</details>

### Q64 — Why the 0.75 power works
Word2vec samples negatives from $P(w) \propto f(w)^{0.75}$. Let's quantify why.

- (a) Compare the raw unigram $P(w) \propto f(w)$ and the smoothed $P(w) \propto f(w)^{0.75}$. What fraction of samples goes to the top 100 words under each?
- (b) What fraction goes to rare words (rank 1000+) under each?
- (c) Why does the $0.75$ compromise work better than either extreme?

In [ ]:
V = 10_000
f = 1 / np.arange(1, V + 1)   # Zipfian
p_raw = f / f.sum()
p_smoothed = f**0.75
p_smoothed /= p_smoothed.sum()
p_uniform = np.ones(V) / V

print(f'                    raw f(w)      f(w)^0.75    uniform')
print(f'Top 100 words:      {p_raw[:100].sum()*100:5.1f}%        '
      f'{p_smoothed[:100].sum()*100:5.1f}%       {p_uniform[:100].sum()*100:5.1f}%')
print(f'Rank 1000+:         {p_raw[1000:].sum()*100:5.1f}%        '
      f'{p_smoothed[1000:].sum()*100:5.1f}%       {p_uniform[1000:].sum()*100:5.1f}%')

# Plot
plt.loglog(p_raw, label='raw f(w)')
plt.loglog(p_smoothed, label='f(w)^0.75')
plt.loglog(p_uniform, label='uniform')
plt.xlabel('rank (log)')
plt.ylabel('sampling probability (log)')
plt.legend()
plt.title('Sampling distributions over 10,000 words')
plt.show()

<details>
<summary><b>Show solution</b></summary>

- (a) Raw $f(w)$: top 100 words get ~70%+ of samples. $f(w)^{0.75}$: closer to ~50%. The $0.75$ exponent **pulls probability mass away from the most frequent words** and spreads it toward the middle and tail.
- (b) Raw $f(w)$: rank-1000+ words get only ~5% of samples. $f(w)^{0.75}$: they get ~15-20%. Rare words still get seen often enough to train.
- (c) Trade-off:
  - **Raw $f$**: too many negatives are "the", "of", "and". These are easy to tell apart from almost any target, so the gradient is tiny. Wasted compute.
  - **Uniform**: too many negatives are rare words that the model has barely seen. Their embeddings are random, so they're also easy to distinguish — again, wasted compute.
  - **$f^{0.75}$**: Mikolov et al. empirically found this specific exponent gives negatives that are **neither too easy nor too hard** — informative gradients across the whole vocabulary. The $3/4$ power doesn't come from theory; it was tuned experimentally.

  (In theory, the "optimal" negative distribution would adaptively re-weight based on current model uncertainty. In practice, $f^{0.75}$ is a static approximation that works well enough.)

</details>

### Q65 — Estimating Zipf exponent from a corpus
Given a frequency count of words in a corpus, how do you **estimate** the Zipf exponent $s$?

In [ ]:
# Simulate a corpus with known Zipf exponent s_true, then recover it
true_s = 1.1
V = 5000
true_freqs = 1 / np.arange(1, V + 1) ** true_s
true_freqs /= true_freqs.sum()

# Sample a corpus of 100k tokens
rng = np.random.default_rng(42)
corpus_indices = rng.choice(V, size=100_000, p=true_freqs)
observed_counts = np.bincount(corpus_indices, minlength=V)
observed_ranked = np.sort(observed_counts)[::-1]      # descending
nonzero = observed_ranked > 0

# Fit log(f) = log(C) - s log(r)
ranks = np.arange(1, nonzero.sum() + 1)
log_r = np.log(ranks)
log_f = np.log(observed_ranked[nonzero])
slope, intercept = np.polyfit(log_r, log_f, 1)

print(f'True Zipf exponent:      {true_s:.3f}')
print(f'Estimated Zipf exponent: {-slope:.3f}   (= -slope of log-log fit)')

plt.loglog(ranks, observed_ranked[nonzero], label='observed')
plt.loglog(ranks, np.exp(intercept) * ranks**slope,
           '--', label=f'fit: s={-slope:.2f}')
plt.xlabel('rank')
plt.ylabel('frequency')
plt.legend()
plt.title('Fitting a Zipf exponent')
plt.grid(True, which='both', alpha=0.3)
plt.show()

<details>
<summary><b>Show solution</b></summary>

The procedure:

1. Count word occurrences, sort frequencies in descending order (giving $f_1, f_2, \ldots, f_V$).
2. Take log of ranks and log of frequencies.
3. Fit a line: $\log f_r = \log C - s \log r$.
4. The estimated Zipf exponent is $-\text{slope}$ of this fit.

In code: `slope, intercept = np.polyfit(log_r, log_f, 1); s = -slope`.

**Caveats**:
- Rare words (bottom of the tail) have very noisy counts because they appear only a handful of times. Many Zipf estimates **truncate** the tail, fitting only to words with count ≥ 5 or so.
- The real distribution of English isn't a pure power law — it's closer to a Zipf-Mandelbrot distribution $f(r) = C / (r + \beta)^s$. Fitting a pure Zipf exponent is a good approximation but not exact.
- English $s \approx 1$; some other languages have slightly different values (e.g., Chinese $s \approx 0.9$).

</details>